# Yêu cầu 2: Model Training & Tracking
- Train 2 model: Naive Bayes & XGBoost
- Split train/val/test
- Đánh giá: Precision, Recall, F1
- Track bằng MLflow (log Parameters, Metrics, Model)

In [ ]:
import joblib
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import precision_score, recall_score, f1_score
from xgboost import XGBClassifier

## 1. Load dữ liệu đã xử lý

In [ ]:
X = joblib.load('../1_DataPipeline/processed/X_tfidf.pkl')
y = joblib.load('../1_DataPipeline/processed/y.pkl')
print('Shape:', X.shape, y.shape)

## 2. Split train / val / test (60/20/20)

In [ ]:
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=y_tmp)
print('Train:', X_train.shape, 'Val:', X_val.shape, 'Test:', X_test.shape)

## 3. Hàm đánh giá

In [ ]:
def evaluate(model, X_, y_):
    pred = model.predict(X_)
    return {
        'precision': precision_score(y_, pred),
        'recall':    recall_score(y_, pred),
        'f1':        f1_score(y_, pred),
    }

## 4. Cấu hình MLflow

In [ ]:
mlflow.set_tracking_uri('file:./mlruns')
mlflow.set_experiment('spam_classification')

## 5. Train Naive Bayes + log MLflow

In [ ]:
with mlflow.start_run(run_name='naive_bayes'):
    params = {'alpha': 1.0}
    nb = MultinomialNB(**params)
    nb.fit(X_train, y_train)
    val_metrics  = evaluate(nb, X_val,  y_val)
    test_metrics = evaluate(nb, X_test, y_test)

    mlflow.log_params(params)
    mlflow.log_metrics({f'val_{k}':  v for k, v in val_metrics.items()})
    mlflow.log_metrics({f'test_{k}': v for k, v in test_metrics.items()})
    mlflow.sklearn.log_model(nb, 'model')
    print('Naive Bayes test:', test_metrics)

## 6. Train XGBoost + log MLflow

In [ ]:
with mlflow.start_run(run_name='xgboost'):
    params = {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1}
    xgb = XGBClassifier(**params, eval_metric='logloss')
    xgb.fit(X_train, y_train)
    val_metrics  = evaluate(xgb, X_val,  y_val)
    test_metrics = evaluate(xgb, X_test, y_test)

    mlflow.log_params(params)
    mlflow.log_metrics({f'val_{k}':  v for k, v in val_metrics.items()})
    mlflow.log_metrics({f'test_{k}': v for k, v in test_metrics.items()})
    mlflow.xgboost.log_model(xgb, 'model')
    print('XGBoost test:', test_metrics)

## 7. Mở MLflow UI để chụp ảnh
Chạy lệnh trong terminal:
```bash
mlflow ui --backend-store-uri file:./mlruns
```
Truy cập http://127.0.0.1:5000, chụp ảnh màn hình so sánh 2 run, lưu vào `mlflow_ui.png`.